In [62]:
!pip install transformers datasets huggingface_hub transformers[torch] accelerate --upgrade

In [63]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch

In [64]:
from huggingface_hub import login

login()

In [65]:
import re
from sklearn.model_selection import train_test_split

In [66]:
f = open("./drive/MyDrive/metriccoders_datasets/history_of_upanishads.txt", "r")
text = f.readlines()

In [67]:
print(len(text))

676


In [68]:
def build_text_files(data_text, dest_path):
    f = open(dest_path, 'w')
    data = ''
    for texts in data_text:
        summary = str(texts).strip()
        summary = re.sub(r"\s", " ", summary)
        data += summary + "  "
    f.write(data)

train, test = train_test_split(text,test_size=0.15)


build_text_files(train,'train_dataset.txt')
build_text_files(test,'test_dataset.txt')

print("Train dataset length: "+str(len(train)))
print("Test dataset length: "+ str(len(test)))

Train dataset length: 574
Test dataset length: 102


In [69]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [70]:
train_path = "train_dataset.txt"
test_path = "test_dataset.txt"

In [71]:
from transformers import TextDataset, DataCollatorForLanguageModeling
model = AutoModelForCausalLM.from_pretrained("gpt2")

In [72]:
def load_dataset(train_path, test_path, tokeinzer):
  train_dataset = TextDataset(
          tokenizer=tokenizer,
          file_path=train_path,
          block_size=64)
  test_dataset = TextDataset(
          tokenizer=tokenizer,
          file_path=test_path,
          block_size=64)
  data_collator = DataCollatorForLanguageModeling(
          tokenizer=tokenizer, mlm=False,
  )
  return train_dataset, test_dataset, data_collator

train_dataset, test_dataset, data_collator = load_dataset(train_path, test_path, tokenizer)

/usr/local/lib/python3.10/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (19299 > 1024). Running this sequence through the model will result in indexing errors


In [73]:
training_args = TrainingArguments(
    output_dir="./gpt2-history-of-upanishads",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_steps=400,
    save_steps=100,
    save_total_limit=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

In [74]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=20, training_loss=3.9775741577148436, metrics={'train_runtime': 14.3662, 'train_samples_per_second': 41.904, 'train_steps_per_second': 1.392, 'total_flos': 19662225408000.0, 'train_loss': 3.9775741577148436, 'epoch': 2.0})

In [75]:
trainer.save_model()

In [76]:
input_text = "Upanishads contain   "
input_ids = tokenizer.encode(input_text, return_tensors="pt").to("cuda")

In [77]:
output = model.generate(input_ids, max_length=100, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [78]:
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

Upanishads contain                                                                                                


In [79]:
from huggingface_hub import notebook_login, create_repo, Repository
notebook_login()


In [80]:
repo_name = "fine-tuned-gpt2-history-of-upanishads"  # Change this to your desired repository name
from huggingface_hub import HfApi

# Initialize the HfApi instance
api = HfApi(token="")

# Create a new repository
username = api.whoami()['name']  # Get your Hugging Face username
full_repo_name = f"{username}/{repo_name}"

# Create the repository (you can also create it on the Hugging Face website)
api.create_repo(repo_name, private=False)

api.upload_folder(
    folder_path='./gpt2-history-of-upanishads',  # Path to the folder with your model
    repo_id=full_repo_name,  # Model repository name
    commit_message="GPT-2 Upanishads"
)

rng_state.pth:   0%|          | 0.00/14.2k [00:00<?, ?B/s]

Upload 8 LFS files:   0%|          | 0/8 [00:00<?, ?it/s]

training_args.bin:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

optimizer.pt:   0%|          | 0.00/996M [00:00<?, ?B/s]

scheduler.pt:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

events.out.tfevents.1724444904.cb2082bc24f6.2441.2:   0%|          | 0.00/5.51k [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/metriccoders/fine-tuned-gpt2-history-of-upanishads/commit/b11d5648b8f4e2f7c9e0796ec611fb45cfc38c6e', commit_message='GPT-2 Upanishads', commit_description='', oid='b11d5648b8f4e2f7c9e0796ec611fb45cfc38c6e', pr_url=None, pr_revision=None, pr_num=None)